# 00 — Setup lingkungan (lokal)

Versi lokal dari Langkah 0 PRD (tanpa Google Colab / Drive).

1. Buat environment sekali dari terminal, di folder `ui3d/`:

```bash
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
jupyter lab
```

2. Jalankan notebook ini: cek library, folder, file data mentah, GPU, dan grid geoid EGM2008.
3. Parameter global ada di `config.yaml` (CRS, path, `GFl`, `Fl`, konstanta +1,0 m, level banjir).

In [1]:
import importlib
import sys

from ui3d_common import CFG, ROOT, raw

print("Python", sys.version.split()[0])
print("Root proyek:", ROOT)

Python 3.10.12
Root proyek: /home/luwes-system/Desktop/bahan ajar 2026/ui3d


## Cek library

In [2]:
LIBS = ["numpy", "pandas", "geopandas", "shapely", "pyproj", "rasterio", "scipy", "skimage", "yaml",
        "matplotlib", "plotly", "trimesh", "mapbox_earcut", "pydeck", "PIL", "torch", "torchvision"]
for name in LIBS:
    try:
        m = importlib.import_module(name)
        print(f"OK   {name:15s} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"MISS {name:15s} {e}")

OK   numpy           2.2.6
OK   pandas          2.3.3
OK   geopandas       1.1.4
OK   shapely         2.1.2
OK   pyproj          3.7.1
OK   rasterio        1.4.4
OK   scipy           1.15.3
OK   skimage         0.25.2
OK   yaml            6.0.3
OK   matplotlib      3.10.9
OK   plotly          7.1.0


OK   trimesh         5.1.1
OK   mapbox_earcut   2.1.0
OK   pydeck          0.9.3
OK   PIL             12.3.0


OK   torch           2.14.1+cu130


OK   torchvision     0.29.1+cu130


## Folder kerja

In [3]:
for key in ("raw", "interim", "output", "models", "figures"):
    d = ROOT / CFG["paths"][key]
    d.mkdir(parents=True, exist_ok=True)
    print(d)

/home/luwes-system/Desktop/bahan ajar 2026/ui3d/raw
/home/luwes-system/Desktop/bahan ajar 2026/ui3d/interim
/home/luwes-system/Desktop/bahan ajar 2026/ui3d/output
/home/luwes-system/Desktop/bahan ajar 2026/ui3d/models
/home/luwes-system/Desktop/bahan ajar 2026/ui3d/figures


## File data mentah

Semua sudah diunduh ke `raw/` (lihat `raw/README.md`). Yang hilang bisa diunduh ulang di sel berikutnya,
kecuali DEMNAS dan Open Buildings 2.5D (butuh akun, unduh manual).

In [4]:
missing = []
for key, rel in CFG["raw"].items():
    ok = raw(key).exists()
    print(("OK  " if ok else "MISS"), key, rel)
    if not ok:
        missing.append(key)
missing

OK   aoi raw/aoi_ui.gpkg
OK   osm raw/osm_ui.gpkg
OK   osm_xml raw/osm_ui_bbox.osm
OK   demnas raw/DEMNAS_1209-42_v1.0.tif
OK   copdem raw/copdem_glo30_ui.tif
OK   icesat2 raw/icesat2_atl08_ui.gpkg
OK   imagery raw/ui_rgb_esri_z19.tif
OK   gob_v3 raw/gob_v3_ui_bbox.gpkg
OK   gob_25d raw/OpenBuildings25D_UI_Depok_2023.tif


[]

In [5]:
# Unduh ulang hanya file yang hilang. Set True untuk menjalankan.
REDOWNLOAD = False

if REDOWNLOAD and missing:
    import geopandas as gpd

    import download as dl

    bbox = tuple(CFG["aoi"]["bbox_fallback"])
    pad = CFG["aoi"]["terrain_buffer_m"] / 111_000  # derajat kira-kira
    bbox_terrain = (bbox[0] - pad, bbox[1] - pad, bbox[2] + pad, bbox[3] + pad)
    if "aoi" in missing:
        dl.aoi(CFG["aoi"]["query"], bbox, raw("aoi"))
    if "osm" in missing or "osm_xml" in missing:
        poly = gpd.read_file(raw("aoi"), layer="aoi_buf50").geometry.iloc[0]
        dl.osm(poly.bounds, raw("osm_xml"), raw("osm"), poly)
    if "copdem" in missing:
        dl.copdem(bbox_terrain, raw("copdem"))
    if "imagery" in missing:
        dl.esri(bbox, CFG["imagery"]["zoom"], raw("imagery"))
    if "imagery_context" in missing:
        dl.esri(bbox_terrain, CFG["imagery"]["context_zoom"], raw("imagery_context"))
    if "gob_v3" in missing:
        dl.gob_v3(bbox, raw("gob_v3"))
    if "icesat2" in missing:
        dl.icesat2(bbox_terrain, raw("icesat2"))
    for key in ("demnas", "gob_25d"):
        if key in missing:
            print(f"{key}: unduh manual (butuh akun), simpan ke {CFG['raw'][key]}")

## GPU (untuk Langkah 4)

Training Mask R-CNN di CPU bisa makan berjam-jam per epoch. Pakai GPU CUDA bila ada.

In [6]:
import torch

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

CUDA: True NVIDIA GeForce RTX 3050


## Grid geoid EGM2008

Konversi tinggi ICESat-2 (ellipsoid) ke EGM2008 butuh grid PROJ `us_nga_egm08_25.tif`.
Sel ini mengaktifkan jaringan PROJ agar grid diunduh otomatis saat dipakai, lalu menguji transformasinya.
Bila tidak ada internet: `projsync --file us_nga_egm08_25.tif`.

In [7]:
import pyproj
from pyproj import Transformer

pyproj.network.set_network_enabled(True)
t = Transformer.from_crs(CFG["crs"]["icesat2_src"], CFG["crs"]["vertical"], always_xy=True)
_, _, h = t.transform(106.826, -6.36, 100.0)
print(f"h ellipsoid 100 m -> EGM2008 {h:.2f} m (undulasi ~{100 - h:.1f} m; harus jauh dari 0)")

h ellipsoid 100 m -> EGM2008 82.04 m (undulasi ~18.0 m; harus jauh dari 0)
